In [ ]:
!unzip transreid_colab.zip

Archive:  transreid_colab.zip
   creating: transreid_colab/
  inflating: transreid_colab/.gitignore  
   creating: transreid_colab/config/
  inflating: transreid_colab/config/defaults.py  
  inflating: transreid_colab/config/__init__.py  
   creating: transreid_colab/config/__pycache__/
  inflating: transreid_colab/config/__pycache__/defaults.cpython-313.pyc  
  inflating: transreid_colab/config/__pycache__/__init__.cpython-313.pyc  
   creating: transreid_colab/configs/
   creating: transreid_colab/configs/DukeMTMC/
  inflating: transreid_colab/configs/DukeMTMC/deit_transreid_stride.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_base.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_jpm.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_sie.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_transreid.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_transreid_384.yml  
  inflating: transreid_colab/configs/DukeMTMC/vit_transreid_stride.yml  
  inflating: 

In [ ]:
%cd transreid_colab/

/content/transreid_colab


In [ ]:
%%shell
#!/usr/bin/env bash
set -euo pipefail

python -m pip install --upgrade pip
python -m pip install -r requirements.txt

mkdir -p ../pretrained
if [ ! -f ../pretrained/jx_vit_base_p16_224-80ecf9dd.pth ]; then
  wget -O ../pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
    https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
fi

mkdir -p ../data
mkdir -p ../logs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
--2026-04-16 03:52:11--  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth [following]
--2026-04-16 03:52:11--  https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/16

In [ ]:
%%shell
gdown --fuzzy "https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ&usp=sharing" -O ../data/Market-1501-v15.09.15.zip
unzip -q ../data/Market-1501-v15.09.15.zip -d ../data
rm -rf ../data/market1501
mv ../data/Market-1501-v15.09.15 ../data/market1501
ls ../data/market1501

Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=a2302a34-66ee-44fe-a6cc-6b92689e9715
To: /content/data/Market-1501-v15.09.15.zip
100% 153M/153M [00:01<00:00, 132MB/s]
bounding_box_test  bounding_box_train  gt_bbox	gt_query  query  readme.txt


In [ ]:
!nproc

12


In [ ]:
!python -c "import torch; print('cuda:', torch.cuda.is_available()); print('count:', torch.cuda.device_count()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

cuda: True
count: 1
NVIDIA A100-SXM4-40GB


In [ ]:
!pip install transformers accelerate

In [ ]:
!pip install schp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [schp]


In [ ]:
!python tools/prepare_semantic_maps.py \
  --dataset-root /content/data/market1501 \
  --output-root /content/data/market1501/semantic_groups \
  --raw-output-root /content/data/market1501/raw_parsing \
  --preset lip6 \
  --subdirs bounding_box_train query bounding_box_test \
  --batch-size 64 \
  --device cuda

config.json: 1.65kB [00:00, 5.58MB/s]
preprocessor_config.json: 100% 340/340 [00:00<00:00, 2.34MB/s]
model.safetensors: 100% 256M/256M [00:03<00:00, 77.9MB/s]
Loading weights: 100% 930/930 [00:01<00:00, 720.86it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
Preparing semantic maps:   0% 0/36036 [00:00<?, ?img/s]/content/transreid_colab/tools/prepare_semantic_maps.py:119: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask_array, mode="L").save(path)
Preparing semantic maps: 100% 36036/36036 [14:04<00:00, 42.67img/s, skipped=0, written=36036]
Model: fashn-ai/fashn-human-parser
Preset: lip6
Device: cuda
Images found: 36036
Grouped masks written: 36036
Skipped existing: 0
Grouped output root: /content/data/market1501/semantic_groups
Raw parsing output root: /content/data/market1501/raw_parsing


In [ ]:
%%shell
python train.py \
  --config_file configs/Market/vit_transreid_stride_sem_align.yml \
  MODEL.DEVICE_ID "'0'" \
  MODEL.PRETRAIN_CHOICE "imagenet" \
  MODEL.PRETRAIN_PATH "/content/pretrained/jx_vit_base_p16_224-80ecf9dd.pth" \
  MODEL.SEM_ALIGN.FLIP_LABEL_PAIRS "[[3,4],[5,6]]" \
  DATASETS.ROOT_DIR "/content/data/" \
  OUTPUT_DIR "runs/market_transreid_sem_align_v2" \
  SOLVER.IMS_PER_BATCH 64 \
  SOLVER.BASE_LR 0.008 \
  DATALOADER.NUM_WORKERS 12

2026-04-16 04:07:33,347 transreid INFO: Saving model in the path :runs/market_transreid_sem_align_v2
2026-04-16 04:07:33,347 transreid INFO: Namespace(config_file='configs/Market/vit_transreid_stride_sem_align.yml', opts=['MODEL.DEVICE_ID', "'0'", 'MODEL.PRETRAIN_CHOICE', 'imagenet', 'MODEL.PRETRAIN_PATH', '/content/pretrained/jx_vit_base_p16_224-80ecf9dd.pth', 'MODEL.SEM_ALIGN.FLIP_LABEL_PAIRS', '[[3,4],[5,6]]', 'DATASETS.ROOT_DIR', '/content/data/', 'OUTPUT_DIR', 'runs/market_transreid_sem_align_v2', 'SOLVER.IMS_PER_BATCH', '64', 'SOLVER.BASE_LR', '0.008', 'DATALOADER.NUM_WORKERS', '12'], local_rank=0)
2026-04-16 04:07:33,347 transreid INFO: Loaded configuration file configs/Market/vit_transreid_stride_sem_align.yml
2026-04-16 04:07:33,347 transreid INFO: 
MODEL:
  PRETRAIN_CHOICE: 'imagenet'
  PRETRAIN_PATH: '/home/heshuting/.cache/torch/checkpoints/jx_vit_base_p16_224-80ecf9dd.pth'
  METRIC_LOSS_TYPE: 'triplet'
  IF_LABELSMOOTH: 'off'
  IF_WITH_CENTER: 'no'
  NAME: 'transformer'
  